# PointTransformerV2 — Point Cloud Segmentation

Beginner-friendly notebook. Runs top to bottom.

⚠️ Requires `torch-scatter` + `torch-cluster` — install matching wheels from https://data.pyg.org/whl/

In [1]:
!pip install dagshub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.4/15.4 MB 416.1 kB/s  0:00:39m0:00:0100:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 472.4 kB/s  0:00:10 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19/19 [dagshub]8/19 [dagshub]]


In [ ]:
import os, io, glob, platform, glob, copy, random, logging
import requests
import numpy as np
import torch
import torch.nn as nn
import open3d as o3d
import laspy
import joblib
import mlflow
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from sklearn.preprocessing import StandardScaler
import dagshub
logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s | %(levelname)s | %(message)s")
log = logging.getLogger(__name__)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
log.info(f"Running on: {DEVICE}")

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


2026-07-20 21:58:13,004 | INFO | Running on: cuda


In [2]:
import io, os, glob, socket, urllib.parse
import gdown
USE_GOOGLE_DRIVE = False
GDRIVE_FOLDER_ID = "https://drive.google.com/drive/folders/1fUCLmHNBI5gCP4w1Z-qA49Lkq4V_8xhC?usp=sharing"
LOCAL_DATA_DIR   = "../data/train_v1"
RAM_STORE = {}
def _load_from_drive(folder_id):
    folder_id = urllib.parse.urlparse(folder_id).path.rstrip("/").split("/")[-1]
    _old_timeout = socket.getdefaulttimeout()
    socket.setdefaulttimeout(60)
    try:
        try:
            entries = gdown.download_folder(id=folder_id, skip_download=True, quiet=True)
        except (TimeoutError, socket.timeout) as e:
            raise RuntimeError(
                f"Timed out listing Drive folder {folder_id!r} after 60s. "
                f"Check your network connection, or that the folder is "
                f"actually shared as 'Anyone with the link'.") from e
    finally:
        socket.setdefaulttimeout(_old_timeout)
    entries = [e for e in entries
              if e.local_path.lower().endswith((".las", ".laz"))]
    if not entries:
        raise RuntimeError(
            "No .las/.laz files found in that folder. Check GDRIVE_FOLDER_ID "
            "and that the folder is shared as 'Anyone with the link'.")
    for e in entries:
        name = os.path.basename(e.local_path)
        buf = io.BytesIO()
        _old_timeout = socket.getdefaulttimeout()
        socket.setdefaulttimeout(60)
        try:
            try:
                gdown.download(id=e.id, output=buf, quiet=True)
            except (TimeoutError, socket.timeout) as ex:
                raise RuntimeError(
                    f"Timed out downloading {name!r} after 60s — Drive may be "
                    f"rate-limiting or the connection stalled. Try again, or "
                    f"download this one file manually to confirm connectivity."
                ) from ex
        finally:
            socket.setdefaulttimeout(_old_timeout)
        buf.seek(0)
        RAM_STORE[name] = buf.getvalue()
        log.info(f"  downloaded {name} "
                 f"({len(RAM_STORE[name]) / (1024**2):.1f} MB) into RAM")
def _list_local_files(folder):
    files = []
    for ext in (".las", ".laz"):
        files += glob.glob(os.path.join(folder, "*" + ext))
    files = sorted(files)
    if not files:
        raise RuntimeError(f"No .las/.laz files found in {folder!r}.")
    log.info(f"Found {len(files)} local file(s) in {folder!r} "
             f"(read directly from disk on demand — not duplicated into RAM)")
    return files
def load_dataset_source():
    if USE_GOOGLE_DRIVE:
        log.info(f"Loading dataset from Google Drive (folder {GDRIVE_FOLDER_ID!r})…")
        _load_from_drive(GDRIVE_FOLDER_ID)
        files = sorted(RAM_STORE.keys())
        total_mb = sum(len(b) for b in RAM_STORE.values()) / (1024**2)
        log.info(f"Loaded {len(files)} files into RAM_STORE ({total_mb:.1f} MB total)")
    else:
        files = _list_local_files(LOCAL_DATA_DIR)
    return files
ALL_FILES = load_dataset_source()

2026-07-20 21:58:13,039 | INFO | Found 45 local file(s) in '../data/train_v1' (read directly from disk on demand — not duplicated into RAM)


In [3]:
CONFIG = {
    "checkpoint_dir" : "../checkpoints/PointCloudDataset_SlidingWindow_10_channel_v2",
    "num_classes"    : 2,
    "target_class"   : 1,
    "val_ratio"      : 0.15,
    "test_ratio"     : 0.15,
    "cv_folds"       : 1,
    "seed"           : 42,
    "epochs"         : 15,
    "patience"       : 1000,
    "batch_size"     : 8,
    "num_points"     : 16384,
    "chunks_per_cloud": 100,
    "lr"             : 2e-4,
    "grad_accum"     : 2,
    "sched_factor"   : 0.5,
    "sched_patience" : 10,
    "min_lr"         : 1e-6,
    "voxel_size"     : 0.01,
    "normal_radius"  : 0.05,
    "normal_max_nn"  : 30,
    "sor_enabled"    : True,
    "sor_k"          : 16,
    "sor_std"        : 2.0,
    "cluster_filter_enabled": True,
    "cluster_eps"    : 0.03,
    "cluster_min_size": 100,
    "aug_jitter"     : 0.01,
    "aug_scale"      : 0.1,
    "aug_dropout"    : 0.10,
    "use_linearity"    : True,
    "use_planarity"    : True,
    "use_sphericity"   : True,
    "use_verticality"  : True,
    "use_eigenentropy" : True,
    "geom_radius"      : 0.06,
    "use_rgb"          : False,
    "in_channels"      : 7,
    "num_workers"    : (0 if platform.system() == "Windows" else 4),
    "weight_decay"        : 1e-4,
    "auto_regularize"     : True,
    "auto_reg_patience"   : 15,
    "auto_reg_wd_factor"  : 2.0,
    "auto_reg_wd_max"     : 1e-2,
    "auto_reg_dropout_step": 0.05,
    "auto_reg_dropout_max": 0.6,
    "auto_reg_max_triggers": 3,
    "use_amp"        : True,
    "class_weights"  : None,
    "max_vis_files"  : 3,
    "mlflow_uri"     : "http://localhost:5000",
    "mlflow_experiment": "PT_v2_SlidingWindow_10_channel",
}
os.makedirs(CONFIG["checkpoint_dir"], exist_ok=True)
CONFIG["in_channels"] = (7 + (1 if CONFIG["use_linearity"] else 0)
                           + (1 if CONFIG["use_planarity"] else 0)
                           + (1 if CONFIG["use_sphericity"] else 0)
                           + (1 if CONFIG["use_verticality"] else 0)
                           + (1 if CONFIG["use_eigenentropy"] else 0)
                           + (3 if CONFIG["use_rgb"] else 0))
NUM_CLASSES = CONFIG["num_classes"]
def compute_class_weights(files, cap=10.0):
    counts = np.zeros(NUM_CLASSES, dtype=np.int64)
    for f in files:
        _, lbl = load_pointcloud(f)
        if lbl is None:
            continue
        lbl = np.clip(lbl, 0, NUM_CLASSES - 1)
        counts += np.bincount(lbl, minlength=NUM_CLASSES)
    if counts.min() == 0:
        return None
    freq = counts / counts.sum()
    w = 1.0 / np.clip(freq, 1e-6, None)
    w = w / w.mean()
    w = np.clip(w, 1.0 / cap, cap)
    return w.tolist()
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
set_seed(CONFIG["seed"])

In [4]:
def load_pointcloud(path, return_rgb=False):
    if path in RAM_STORE:
        las = laspy.read(io.BytesIO(RAM_STORE[path]))
    else:
        las = laspy.read(path)
    pts = np.column_stack([np.asarray(las.x),
                           np.asarray(las.y),
                           np.asarray(las.z)]).astype(np.float64)
    labels = None
    for key in ("classification", "label", "labels", "class"):
        if key in las.point_format.dimension_names:
            labels = np.asarray(getattr(las, key), dtype=np.int64)
            break
    rgb = None
    if return_rgb:
        dims = las.point_format.dimension_names
        if all(k in dims for k in ("red", "green", "blue")):
            rgb = np.column_stack([np.asarray(las.red),
                                   np.asarray(las.green),
                                   np.asarray(las.blue)]).astype(np.float32)
            if rgb.max() > 255:
                rgb /= 65535.0
            elif rgb.max() > 1:
                rgb /= 255.0
    finite = np.isfinite(pts).all(axis=1)
    if not finite.all():
        n_bad = int((~finite).sum())
        log.warning(f"{os.path.basename(path)}: dropping {n_bad} non-finite points")
        pts = pts[finite]
        if labels is not None:
            labels = labels[finite]
        if rgb is not None:
            rgb = rgb[finite]
    if labels is not None:
        assert len(labels) == len(pts), \
            f"{os.path.basename(path)}: label/point count mismatch"
    if return_rgb:
        return pts, rgb, labels
    return pts, labels
from sklearn.model_selection import KFold
all_files = sorted(ALL_FILES)
assert all_files, "ALL_FILES is empty — did load_dataset_source() run successfully?"
rng   = np.random.RandomState(CONFIG["seed"])
order = rng.permutation(len(all_files))
n_test = max(1, int(len(all_files) * CONFIG["test_ratio"]))
TEST_FILES = [all_files[i] for i in order[:n_test]]
remaining  = [all_files[i] for i in order[n_test:]]
if CONFIG.get("cv_folds", 1) <= 1:
    n_val = max(1, int(len(all_files) * CONFIG["val_ratio"]))
    VAL_FILES   = remaining[:n_val]
    TRAIN_FILES = remaining[n_val:]
    CV_FOLDS = None
    log.info(f"Single split: train={len(TRAIN_FILES)}  val={len(VAL_FILES)}  "
             f"test={len(TEST_FILES)}")
else:
    kf = KFold(n_splits=CONFIG["cv_folds"], shuffle=True,
              random_state=CONFIG["seed"])
    CV_FOLDS = []
    for tr_idx, va_idx in kf.split(remaining):
        CV_FOLDS.append(([remaining[i] for i in tr_idx],
                         [remaining[i] for i in va_idx]))
    TRAIN_FILES, VAL_FILES = CV_FOLDS[0]
    log.info(f"{CONFIG['cv_folds']}-fold CV on {len(remaining)} files "
             f"(test={len(TEST_FILES)} held out, never in any fold)")
    for i, (tf, vf) in enumerate(CV_FOLDS):
        log.info(f"  fold {i}: train={len(tf)}  val={len(vf)}")
INFER_FILES = []
log.info(f"train={len(TRAIN_FILES)}  val={len(VAL_FILES)}  "
         f"test={len(TEST_FILES)}  inference={len(INFER_FILES)}")

2026-07-20 21:58:13,068 | INFO | Single split: train=33  val=6  test=6
2026-07-20 21:58:13,068 | INFO | train=33  val=6  test=6  inference=0


In [5]:
USE_GOOGLE_DRIVE_INFER = False
GDRIVE_INFER_FOLDER_ID = "https://drive.google.com/drive/folders/1XBszFp5F7ZP-ucTAf-Pp8hdQhPHHuMn1?usp=sharing"
LOCAL_INFER_DIR        = "../data/test"
def load_inference_files():
    try:
        if USE_GOOGLE_DRIVE_INFER:
            log.info(f"Loading inference set from Google Drive "
                     f"(folder {GDRIVE_INFER_FOLDER_ID!r})…")
            before = set(RAM_STORE.keys())
            _load_from_drive(GDRIVE_INFER_FOLDER_ID)
            new_keys = sorted(set(RAM_STORE.keys()) - before)
            log.info(f"Loaded {len(new_keys)} inference file(s) into RAM_STORE "
                     f"(kept separate from train/val/test)")
            return new_keys
        else:
            log.info(f"Loading inference set from local disk "
                     f"({LOCAL_INFER_DIR!r})…")
            return _list_local_files(LOCAL_INFER_DIR)
    except RuntimeError as e:
        log.warning(f"No inference set loaded ({e}) — INFER_FILES will be empty.")
        return []
INFER_FILES = load_inference_files()

2026-07-20 21:58:13,080 | INFO | Loading inference set from local disk ('../data/test')…
2026-07-20 21:58:13,081 | INFO | Found 18 local file(s) in '../data/test' (read directly from disk on demand — not duplicated into RAM)


In [6]:
def _geom_features(points, radius=0.06, min_neighbors=8,
                   want_linear=True, want_planar=False, want_sphere=False,
                   want_vert=False, want_entropy=False):
    from scipy.spatial import cKDTree as _GeomTree
    n = len(points)
    lin = np.zeros(n, np.float32) if want_linear else None
    pla = np.zeros(n, np.float32) if want_planar else None
    sph = np.zeros(n, np.float32) if want_sphere else None
    vert = np.zeros(n, np.float32) if want_vert else None
    ent  = np.zeros(n, np.float32) if want_entropy else None
    if not (want_linear or want_planar or want_sphere or want_vert or want_entropy):
        return lin, pla, sph, vert, ent
    need_evecs = want_vert
    tree = _GeomTree(points)
    lists = tree.query_ball_point(points, r=radius, workers=-1)
    for i, nbrs in enumerate(lists):
        if len(nbrs) < min_neighbors:
            continue
        nb = points[nbrs]
        c = nb - nb.mean(0)
        cov = (c.T @ c) / len(nbrs)
        if need_evecs:
            evals, evecs = np.linalg.eigh(cov)
            l3, l2, l1 = evals
        else:
            l3, l2, l1 = np.linalg.eigvalsh(cov)
        l1c = max(l1, 1e-9)
        if want_linear:  lin[i] = (l1 - l2) / l1c
        if want_planar:  pla[i] = (l2 - l3) / l1c
        if want_sphere:  sph[i] = l3 / l1c
        if want_vert:
            normal_proxy = evecs[:, 0]
            vert[i] = 1.0 - abs(normal_proxy[2])
        if want_entropy:
            s = l1 + l2 + l3
            if s > 1e-9:
                p = np.clip(np.array([l1, l2, l3]) / s, 1e-12, None)
                ent[i] = -(p * np.log(p)).sum()
    return lin, pla, sph, vert, ent
def make_features(points, rgb=None):
    center = points.mean(axis=0, keepdims=True)
    scale  = max(np.linalg.norm(points - center, axis=1).max(), 1e-9)
    norm_xyz = ((points - center) / scale).astype(np.float32)
    z = points[:, 2]
    height = ((z - z.min()) / max(z.max() - z.min(), 1e-6)).astype(np.float32)
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(points.astype(np.float64))
    pcd.estimate_normals(o3d.geometry.KDTreeSearchParamHybrid(
        radius=CONFIG.get("normal_radius", 0.05),
        max_nn=CONFIG.get("normal_max_nn", 30)))
    pcd.orient_normals_to_align_with_direction([0., 0., 1.])
    normals = np.asarray(pcd.normals, dtype=np.float32)
    cols = [norm_xyz, height[:, None], normals]
    want_lin = CONFIG.get("use_linearity", False)
    want_pla = CONFIG.get("use_planarity", False)
    want_sph = CONFIG.get("use_sphericity", False)
    want_vert = CONFIG.get("use_verticality", False)
    want_ent  = CONFIG.get("use_eigenentropy", False)
    if want_lin or want_pla or want_sph or want_vert or want_ent:
        lin, pla, sph, vert, ent = _geom_features(
            points, radius=CONFIG.get("geom_radius", 0.06),
            want_linear=want_lin, want_planar=want_pla, want_sphere=want_sph,
            want_vert=want_vert, want_entropy=want_ent)
        if want_lin:  cols.append(lin[:, None])
        if want_pla:  cols.append(pla[:, None])
        if want_sph:  cols.append(sph[:, None])
        if want_vert: cols.append(vert[:, None])
        if want_ent:  cols.append(ent[:, None])
    if CONFIG.get("use_rgb", False):
        if rgb is None:
            log.warning("use_rgb=True but file has no RGB — filling zeros")
            rgb = np.zeros((len(points), 3), np.float32)
        cols.append(rgb.astype(np.float32))
    feat = np.column_stack(cols)
    assert feat.shape[1] == CONFIG["in_channels"], \
        f"feature dims {feat.shape[1]} != CONFIG in_channels {CONFIG['in_channels']}"
    return feat
def _voxel_keep(pts, voxel):
    vox = np.floor(pts / voxel).astype(np.int64)
    _, inv, cnt = np.unique(vox, axis=0, return_inverse=True, return_counts=True)
    sums = np.zeros((cnt.size, 3), np.float64); np.add.at(sums, inv, pts)
    d2 = ((pts - (sums / cnt[:, None])[inv]) ** 2).sum(1)
    order = np.lexsort((d2, inv))
    first = np.concatenate([[0], np.cumsum(cnt)[:-1]])
    return np.sort(order[first])
def _sor_keep(pts, k=16, std_ratio=2.0):
    from scipy.spatial import cKDTree as _SORTree
    n = len(pts)
    if n <= k + 1:
        return np.ones(n, bool)
    d, _ = _SORTree(pts).query(pts, k=k + 1)
    mean_dist = d[:, 1:].mean(axis=1)
    thr = mean_dist.mean() + std_ratio * mean_dist.std()
    return mean_dist <= thr
def _cluster_keep(pts, eps, min_samples=8, min_cluster_size=100):
    from sklearn.cluster import DBSCAN as _DBSCAN
    if len(pts) < min_samples:
        return np.ones(len(pts), bool)
    labels = _DBSCAN(eps=eps, min_samples=min_samples, n_jobs=-1).fit_predict(pts)
    keep = np.zeros(len(pts), bool)
    for lab in np.unique(labels):
        if lab == -1:
            continue
        idx = np.flatnonzero(labels == lab)
        if len(idx) >= min_cluster_size:
            keep[idx] = True
    return keep
FEATURE_CACHE = {}
def _prep_key(path):
    return (path,
            round(float(CONFIG.get("voxel_size", 0)), 6),
            round(float(CONFIG.get("normal_radius", 0.05)), 6),
            int(CONFIG.get("normal_max_nn", 30)),
            bool(CONFIG.get("sor_enabled", True)),
            round(float(CONFIG.get("sor_std", 2.0)), 3),
            int(CONFIG.get("sor_k", 16)),
            bool(CONFIG.get("cluster_filter_enabled", True)),
            round(float(CONFIG.get("cluster_eps", 0.03)), 4),
            int(CONFIG.get("cluster_min_size", 100)),
            bool(CONFIG.get("use_linearity", False)),
            bool(CONFIG.get("use_planarity", False)),
            bool(CONFIG.get("use_sphericity", False)),
            bool(CONFIG.get("use_verticality", False)),
            bool(CONFIG.get("use_eigenentropy", False)),
            round(float(CONFIG.get("geom_radius", 0.06)), 4),
            bool(CONFIG.get("use_rgb", False)))
def get_features(path):
    key = _prep_key(path)
    if key not in FEATURE_CACHE:
        pts, rgb, lbl = load_pointcloud(path, return_rgb=True)
        lbl = np.zeros(len(pts), dtype=np.int64) if lbl is None else lbl
        lbl = np.clip(lbl, 0, NUM_CLASSES - 1).astype(np.int64)
        v = CONFIG.get("voxel_size", 0)
        if v and v > 0 and len(pts) > 0:
            keep = _voxel_keep(pts, v)
            pts, lbl = pts[keep], lbl[keep]
            if rgb is not None: rgb = rgb[keep]
        if CONFIG.get("sor_enabled", True) and len(pts) > 0:
            keep = _sor_keep(pts, k=CONFIG.get("sor_k", 16),
                             std_ratio=CONFIG.get("sor_std", 2.0))
            n_removed = int((~keep).sum())
            if n_removed > 0:
                log.info(f"  SOR: removed {n_removed} outlier pts "
                         f"({os.path.basename(path)})")
            pts, lbl = pts[keep], lbl[keep]
            if rgb is not None: rgb = rgb[keep]
        if CONFIG.get("cluster_filter_enabled", True) and len(pts) > 0:
            keep = _cluster_keep(pts,
                                 eps=CONFIG.get("cluster_eps", 0.03),
                                 min_samples=8,
                                 min_cluster_size=CONFIG.get("cluster_min_size", 100))
            n_removed = int((~keep).sum())
            if n_removed > 0:
                log.info(f"  Cluster-filter: removed {n_removed} pts in small "
                         f"floating clusters ({os.path.basename(path)})")
            pts, lbl = pts[keep], lbl[keep]
            if rgb is not None: rgb = rgb[keep]
        feat = make_features(pts, rgb)
        FEATURE_CACHE[key] = (pts, feat, lbl)
    return FEATURE_CACHE[key]
log.info("Pre-computing features for train + val files (one-time cost)...")
for f in tqdm(TRAIN_FILES + VAL_FILES, desc="features"):
    get_features(f)
log.info("Done.")

2026-07-20 21:58:13,104 | INFO | Pre-computing features for train + val files (one-time cost)...


features:   0%|          | 0/39 [00:00<?, ?it/s]

2026-07-20 21:58:13,837 | INFO |   SOR: removed 4496 outlier pts (sample_data_0039.las)
2026-07-20 21:58:14,499 | INFO |   Cluster-filter: removed 1424 pts in small floating clusters (sample_data_0039.las)
2026-07-20 21:58:21,235 | INFO |   SOR: removed 6441 outlier pts (sample_data_0027.las)
2026-07-20 21:58:22,246 | INFO |   Cluster-filter: removed 1919 pts in small floating clusters (sample_data_0027.las)
2026-07-20 21:58:31,320 | INFO |   SOR: removed 3241 outlier pts (sample_data_0025.las)
2026-07-20 21:58:31,936 | INFO |   Cluster-filter: removed 1319 pts in small floating clusters (sample_data_0025.las)
2026-07-20 21:58:38,094 | INFO |   SOR: removed 5697 outlier pts (sample_data_0021.las)
2026-07-20 21:58:38,871 | INFO |   Cluster-filter: removed 1570 pts in small floating clusters (sample_data_0021.las)
2026-07-20 21:58:46,372 | INFO |   SOR: removed 5525 outlier pts (sample_data_007.las)
2026-07-20 21:58:47,281 | INFO |   Cluster-filter: removed 2314 pts in small floating clu

In [7]:
def compute_miou(true_labels, pred_labels, num_classes):
    ious = []
    for c in range(num_classes):
        tp = int(((true_labels == c) & (pred_labels == c)).sum())
        fp = int(((true_labels != c) & (pred_labels == c)).sum())
        fn = int(((true_labels == c) & (pred_labels != c)).sum())
        if tp + fp + fn > 0:
            ious.append(tp / (tp + fp + fn))
    return float(np.mean(ious)) if ious else 0.0
def compute_dice(true_labels, pred_labels, num_classes):
    dices = []
    for c in range(num_classes):
        tp = int(((true_labels == c) & (pred_labels == c)).sum())
        fp = int(((true_labels != c) & (pred_labels == c)).sum())
        fn = int(((true_labels == c) & (pred_labels != c)).sum())
        if tp + fp + fn > 0:
            dices.append((2 * tp) / (2 * tp + fp + fn))
    return float(np.mean(dices)) if dices else 0.0
def per_file_report(names, true_list, pred_list, tag="", target_class=1):
    header = (f"{'File':<22}{'IoU':>7}{'Dice':>7}{'Prec':>7}{'Rec':>7}"
             f"{'TP':>9}{'FP':>9}{'FN':>9}{'TN':>9}{'GT_Wood':>10}{'Pred_Wood':>11}")
    print(f"\n{'='*len(header)}")
    print(f"PER-FILE SEGMENTATION REPORT{' — ' + tag if tag else ''}")
    print('='*len(header))
    print(header)
    print('-'*len(header))
    ious, dices, precs, recs = [], [], [], []
    for name, true, pred in zip(names, true_list, pred_list):
        pred_wood = int((pred == target_class).sum())
        if true is None:
            print(f"{name:<22}{'—':>7}{'—':>7}{'—':>7}{'—':>7}"
                 f"{'—':>9}{'—':>9}{'—':>9}{'—':>9}{'—':>10}{pred_wood:>11,}")
            continue
        tp = int(((true == target_class) & (pred == target_class)).sum())
        fp = int(((true != target_class) & (pred == target_class)).sum())
        fn = int(((true == target_class) & (pred != target_class)).sum())
        tn = int(((true != target_class) & (pred != target_class)).sum())
        gt_wood = int((true == target_class).sum())
        iou  = tp / (tp + fp + fn) if (tp + fp + fn) > 0 else 0.0
        dice = (2 * tp) / (2 * tp + fp + fn) if (2 * tp + fp + fn) > 0 else 0.0
        prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        ious.append(iou); dices.append(dice); precs.append(prec); recs.append(rec)
        print(f"{name:<22}{iou:7.3f}{dice:7.3f}{prec:7.3f}{rec:7.3f}"
             f"{tp:9,}{fp:9,}{fn:9,}{tn:9,}{gt_wood:10,}{pred_wood:11,}")
    if ious:
        print('-'*len(header))
        print(f"{'MEAN':<22}{np.mean(ious):7.3f}{np.mean(dices):7.3f}"
             f"{np.mean(precs):7.3f}{np.mean(recs):7.3f}")
    print('='*len(header))

In [ ]:
try:
    dagshub.init(repo_owner='darwald2024', repo_name='ojdo', mlflow=True)
    # mlflow.set_tracking_uri(CONFIG["mlflow_uri"])
    # mlflow.set_experiment(CONFIG["mlflow_experiment"])
    MLFLOW_OK = True
    log.info(f"MLflow tracking: {CONFIG['mlflow_uri']}")
except Exception as e:
    MLFLOW_OK = False
    log.warning(f"MLflow not available ({e}) — training continues without logging")

2026/07/20 22:04:13 INFO mlflow.tracking.fluent: Experiment with name 'PT_v2_SlidingWindow_10_channel' does not exist. Creating a new experiment.
2026-07-20 22:04:13,824 | INFO | MLflow tracking: http://localhost:5000


## Model Architecture

In [9]:
MODEL_NAME = "PointTransformerV2"
from torch_cluster import knn as tc_knn
from torch_scatter import scatter_softmax, scatter_add, scatter_max, scatter_mean
class GVA(nn.Module):
    def __init__(self, ch, g=6, k=16):
        super().__init__()
        assert ch % g == 0
        self.k, self.g, self.gc = k, g, ch // g
        self.q  = nn.Linear(ch, ch); self.kk = nn.Linear(ch, ch); self.v = nn.Linear(ch, ch)
        self.pm = nn.Sequential(nn.Linear(3,ch), nn.ReLU(), nn.Linear(ch,ch))
        self.pb = nn.Sequential(nn.Linear(3,ch), nn.ReLU(), nn.Linear(ch,ch))
        self.w  = nn.Sequential(nn.Linear(ch,ch), nn.ReLU(), nn.Linear(ch,g))
    def forward(self, x, pos, batch):
        e    = tc_knn(pos, pos, self.k, batch, batch)
        c, nb = e[0], e[1]
        dp   = pos[c] - pos[nb]
        pb   = self.pb(dp)
        rel  = (self.q(x)[c] - self.kk(x)[nb]) * self.pm(dp) + pb
        wt   = scatter_softmax(self.w(rel), c, dim=0)
        vg   = (self.v(x)[nb] + pb).view(-1, self.g, self.gc)
        out  = scatter_add(vg * wt.unsqueeze(-1), c, dim=0, dim_size=x.size(0))
        return out.view(-1, self.g * self.gc)
class Block(nn.Module):
    def __init__(self, ch, g=6, k=16):
        super().__init__()
        self.n1, self.n2 = nn.LayerNorm(ch), nn.LayerNorm(ch)
        self.attn = GVA(ch, g, k)
        self.mlp  = nn.Sequential(nn.Linear(ch,ch*2), nn.ReLU(), nn.Linear(ch*2,ch))
    def forward(self, x, pos, batch):
        x = x + self.attn(self.n1(x), pos, batch)
        return x + self.mlp(self.n2(x))
class GridPool(nn.Module):
    def __init__(self, i, o, grid):
        super().__init__()
        self.grid = grid
        self.proj = nn.Sequential(nn.Linear(i,o), nn.ReLU())
    def forward(self, x, pos, batch):
        vox = torch.floor(pos/self.grid).long()
        key = torch.cat([batch.unsqueeze(1), vox], 1)
        _, cl = torch.unique(key, dim=0, return_inverse=True)
        xp, _ = scatter_max(self.proj(x), cl, dim=0)
        return xp, scatter_mean(pos,cl,dim=0), scatter_max(batch,cl,dim=0)[0], cl
class PTv2Seg(nn.Module):
    def __init__(self, nc=NUM_CLASSES, k=16, dims=(48,96,192), g=6):
        super().__init__()
        d = dims
        self.embed = nn.Sequential(
            nn.Linear(CONFIG["in_channels"], d[0]), nn.ReLU(),
            nn.Linear(d[0], d[0]))
        self.e1=Block(d[0],g,k); self.p1=GridPool(d[0],d[1],0.08)
        self.e2=Block(d[1],g,k); self.p2=GridPool(d[1],d[2],0.16)
        self.e3=Block(d[2],g,k)
        self.u2=nn.Sequential(nn.Linear(d[2]+d[1],d[1]),nn.ReLU()); self.d2=Block(d[1],g,k)
        self.u1=nn.Sequential(nn.Linear(d[1]+d[0],d[0]),nn.ReLU()); self.d1=Block(d[0],g,k)
        self.head=nn.Sequential(nn.LayerNorm(d[0]),nn.Linear(d[0],128),nn.ReLU(),
                                nn.Dropout(0.4),nn.Linear(128,nc))
    def forward(self, x):
        B,C,N = x.shape
        flat  = x.permute(0,2,1).reshape(-1,C).contiguous()
        p0    = flat[:,:3].contiguous()
        b0    = torch.arange(B,device=x.device).repeat_interleave(N)
        h0=self.e1(self.embed(flat),p0,b0)
        h1,p1,b1,c1=self.p1(h0,p0,b0); h1=self.e2(h1,p1,b1)
        h2,p2,b2,c2=self.p2(h1,p1,b1); h2=self.e3(h2,p2,b2)
        u1=self.d2(self.u2(torch.cat([h1,h2[c2]],1)),p1,b1)
        u0=self.d1(self.u1(torch.cat([h0,u1[c1]],1)),p0,b0)
        return self.head(u0).view(B,N,-1).permute(0,2,1)
def build_model():
    return PTv2Seg()
model = build_model()
log.info(f"PTv2 parameters: {sum(p.numel() for p in model.parameters()):,}")

2026-07-20 22:04:13,882 | INFO | PTv2 parameters: 679,664


## Dataset & DataLoader

In [10]:
from torch.utils.data import Dataset, DataLoader
def _augment_and_pack(feat, lbl, chosen, N, augment):
    x = feat[chosen].copy()
    y = lbl[chosen].copy()
    x[:, :3] -= x[:, :3].mean(axis=0, keepdims=True)
    if augment:
        angle = np.random.uniform(0, 2 * np.pi)
        c, sn = np.cos(angle), np.sin(angle)
        R = np.array([[c, -sn, 0], [sn, c, 0], [0, 0, 1]], np.float32)
        x[:, :3]  = x[:, :3]  @ R.T
        x[:, 4:7] = x[:, 4:7] @ R.T
        if CONFIG.get("aug_jitter", 0) > 0:
            x[:, :3] += np.random.normal(
                0, CONFIG["aug_jitter"], x[:, :3].shape).astype(np.float32)
        if CONFIG.get("aug_scale", 0) > 0:
            s_ = np.float32(1.0 + np.random.uniform(
                -CONFIG["aug_scale"], CONFIG["aug_scale"]))
            x[:, :3] *= s_
        if CONFIG.get("aug_dropout", 0) > 0:
            keep = np.random.rand(N) > CONFIG["aug_dropout"]
            if keep.sum() > N // 2:
                kept = np.flatnonzero(keep)
                fill = np.random.choice(kept, N - len(kept), replace=True)
                order = np.r_[kept, fill]
                x, y = x[order], y[order]
    return torch.from_numpy(x.T).contiguous(), torch.from_numpy(y)
def _knn_sphere_fallback(feat, rng, N, n):
    seed = rng.randint(n)
    dist = ((feat[:, :3] - feat[seed, :3]) ** 2).sum(1)
    return np.argpartition(dist, N - 1)[:N]
class PointCloudDataset_RandomGrid(Dataset):
    def __init__(self, files, augment=True):
        self.files, self.augment = list(files), augment
        self.N, self.chunks = CONFIG["num_points"], CONFIG["chunks_per_cloud"]
    def __len__(self):
        return len(self.files) * self.chunks
    def __getitem__(self, idx):
        path = self.files[idx // self.chunks]
        pts, feat, lbl = get_features(path)
        n = len(feat)
        rng = np.random if self.augment else np.random.RandomState(idx)
        if n <= self.N:
            chosen = rng.choice(n, self.N, replace=True)
        else:
            col = max(np.ptp(feat[:, 0]), np.ptp(feat[:, 1])) / 6.0 + 1e-9
            cx = rng.uniform(feat[:, 0].min(), feat[:, 0].max())
            cy = rng.uniform(feat[:, 1].min(), feat[:, 1].max())
            pool = np.flatnonzero((np.abs(feat[:, 0] - cx) < col) &
                                  (np.abs(feat[:, 1] - cy) < col))
            if len(pool) < self.N // 4:
                chosen = _knn_sphere_fallback(feat, rng, self.N, n)
            else:
                chosen = rng.choice(pool, self.N, replace=len(pool) < self.N)
        return _augment_and_pack(feat, lbl, chosen, self.N, self.augment)
class PointCloudDataset_SequentialGrid(Dataset):
    GRID = 6
    def __init__(self, files, augment=True):
        self.files, self.augment = list(files), augment
        self.N, self.chunks = CONFIG["num_points"], CONFIG["chunks_per_cloud"]
    def __len__(self):
        return len(self.files) * self.chunks
    def __getitem__(self, idx):
        path = self.files[idx // self.chunks]
        cell_idx = idx % self.chunks
        pts, feat, lbl = get_features(path)
        n = len(feat)
        rng = np.random if self.augment else np.random.RandomState(idx)
        if n <= self.N:
            chosen = rng.choice(n, self.N, replace=True)
        else:
            g = self.GRID
            gx, gy = (cell_idx % (g * g)) % g, (cell_idx % (g * g)) // g
            x0, x1 = feat[:, 0].min(), feat[:, 0].max()
            y0, y1 = feat[:, 1].min(), feat[:, 1].max()
            cx = x0 + (gx + 0.5) * (x1 - x0) / g
            cy = y0 + (gy + 0.5) * (y1 - y0) / g
            col = max(x1 - x0, y1 - y0) / g + 1e-9
            pool = np.flatnonzero((np.abs(feat[:, 0] - cx) < col) &
                                  (np.abs(feat[:, 1] - cy) < col))
            if len(pool) < self.N // 4:
                chosen = _knn_sphere_fallback(feat, rng, self.N, n)
            else:
                chosen = rng.choice(pool, self.N, replace=len(pool) < self.N)
        return _augment_and_pack(feat, lbl, chosen, self.N, self.augment)
class PointCloudDataset_SlidingWindow(Dataset):
    def __init__(self, files, augment=True, window_frac=1/6, overlap=0.5):
        self.files, self.augment = list(files), augment
        self.N, self.chunks = CONFIG["num_points"], CONFIG["chunks_per_cloud"]
        self.window_frac, self.overlap = window_frac, overlap
    def __len__(self):
        return len(self.files) * self.chunks
    def __getitem__(self, idx):
        path = self.files[idx // self.chunks]
        step_idx = idx % self.chunks
        pts, feat, lbl = get_features(path)
        n = len(feat)
        rng = np.random if self.augment else np.random.RandomState(idx)
        if n <= self.N:
            chosen = rng.choice(n, self.N, replace=True)
        else:
            x0, x1 = feat[:, 0].min(), feat[:, 0].max()
            y0, y1 = feat[:, 1].min(), feat[:, 1].max()
            win = max(x1 - x0, y1 - y0) * self.window_frac + 1e-9
            stride = win * (1 - self.overlap)
            n_steps_x = max(int((x1 - x0) / stride), 1)
            sx, sy = step_idx % n_steps_x, step_idx // n_steps_x
            cx = x0 + win / 2 + sx * stride
            cy = y0 + win / 2 + (sy % max(int((y1 - y0) / stride), 1)) * stride
            pool = np.flatnonzero((np.abs(feat[:, 0] - cx) < win / 2) &
                                  (np.abs(feat[:, 1] - cy) < win / 2))
            if len(pool) < self.N // 4:
                chosen = _knn_sphere_fallback(feat, rng, self.N, n)
            else:
                chosen = rng.choice(pool, self.N, replace=len(pool) < self.N)
        return _augment_and_pack(feat, lbl, chosen, self.N, self.augment)
class PointCloudDataset_CoverageBased(Dataset):
    def __init__(self, files, augment=True):
        self.files, self.augment = list(files), augment
        self.N, self.chunks = CONFIG["num_points"], CONFIG["chunks_per_cloud"]
        self._visits = {}
    def __len__(self):
        return len(self.files) * self.chunks
    def __getitem__(self, idx):
        path = self.files[idx // self.chunks]
        pts, feat, lbl = get_features(path)
        n = len(feat)
        rng = np.random if self.augment else np.random.RandomState(idx)
        if path not in self._visits:
            self._visits[path] = np.zeros(n, np.int32)
        visits = self._visits[path]
        if n <= self.N:
            chosen = rng.choice(n, self.N, replace=True)
        else:
            least_visited = np.flatnonzero(visits == visits.min())
            seed = least_visited[rng.randint(len(least_visited))]
            col = max(np.ptp(feat[:, 0]), np.ptp(feat[:, 1])) / 6.0 + 1e-9
            cx, cy = feat[seed, 0], feat[seed, 1]
            pool = np.flatnonzero((np.abs(feat[:, 0] - cx) < col) &
                                  (np.abs(feat[:, 1] - cy) < col))
            if len(pool) < self.N // 4:
                chosen = _knn_sphere_fallback(feat, rng, self.N, n)
            else:
                chosen = rng.choice(pool, self.N, replace=len(pool) < self.N)
        visits[chosen] += 1
        return _augment_and_pack(feat, lbl, chosen, self.N, self.augment)
class PointCloudDataset_FPSCenters(Dataset):
    def __init__(self, files, augment=True):
        self.files, self.augment = list(files), augment
        self.N, self.chunks = CONFIG["num_points"], CONFIG["chunks_per_cloud"]
        self._center_cache = {}
    def __len__(self):
        return len(self.files) * self.chunks
    def __getitem__(self, idx):
        path = self.files[idx // self.chunks]
        pts, feat, lbl = get_features(path)
        n = len(feat)
        rng = np.random if self.augment else np.random.RandomState(idx)
        if n <= self.N:
            chosen = rng.choice(n, self.N, replace=True)
        else:
            if path not in self._center_cache:
                xy = feat[:, :2]
                k = min(self.chunks, n)
                idx0 = np.random.RandomState(0).randint(n)
                d2 = ((xy - xy[idx0]) ** 2).sum(1)
                sel = np.empty(k, np.int64); sel[0] = idx0
                for i in range(1, k):
                    sel[i] = int(d2.argmax())
                    d2 = np.minimum(d2, ((xy - xy[sel[i]]) ** 2).sum(1))
                self._center_cache[path] = xy[sel]
            centers = self._center_cache[path]
            col = max(np.ptp(feat[:, 0]), np.ptp(feat[:, 1])) / 6.0 + 1e-9
            c = centers[rng.randint(len(centers))]
            cx = c[0] + rng.normal(0, col * 0.3)
            cy = c[1] + rng.normal(0, col * 0.3)
            pool = np.flatnonzero((np.abs(feat[:, 0] - cx) < col) &
                                  (np.abs(feat[:, 1] - cy) < col))
            if len(pool) < self.N // 4:
                chosen = _knn_sphere_fallback(feat, rng, self.N, n)
            else:
                chosen = rng.choice(pool, self.N, replace=len(pool) < self.N)
        return _augment_and_pack(feat, lbl, chosen, self.N, self.augment)
PointCloudDataset = PointCloudDataset_SlidingWindow

## Training

In [11]:
CONFIG["class_weights"] = compute_class_weights(TRAIN_FILES)

In [12]:
SAVE_EVERY = 10
LAST_RUN_ID = None
def train_model(model, model_name):
    ckpt_dir  = CONFIG["checkpoint_dir"]
    best_path = os.path.join(ckpt_dir, f"{model_name}_best.pth")
    def latest_periodic():
        pattern = os.path.join(ckpt_dir, f"{model_name}_epoch_*.pth")
        files   = sorted(glob.glob(pattern))
        if not files:
            return 0, None
        def epoch_of(p):
            try:   return int(os.path.basename(p).split("_epoch_")[1].replace(".pth",""))
            except: return 0
        files.sort(key=epoch_of)
        return epoch_of(files[-1]), files[-1]
    def prune_old_checkpoints(keep=2):
        pattern = os.path.join(ckpt_dir, f"{model_name}_epoch_*.pth")
        files   = sorted(glob.glob(pattern))
        def epoch_of(p):
            try:   return int(os.path.basename(p).split("_epoch_")[1].replace(".pth",""))
            except: return 0
        files.sort(key=epoch_of)
        for old in files[:-keep]:
            try:    os.remove(old); log.info(f"Removed old checkpoint: {old}")
            except: pass
    _nw = CONFIG.get("num_workers", 4)
    train_loader = DataLoader(
        PointCloudDataset(TRAIN_FILES, augment=True),
        batch_size=CONFIG["batch_size"], shuffle=True, drop_last=True,
        num_workers=_nw, pin_memory=True, persistent_workers=(_nw > 0))
    val_loader = DataLoader(
        PointCloudDataset(VAL_FILES, augment=False),
        batch_size=CONFIG["batch_size"], shuffle=False,
        num_workers=_nw, pin_memory=True, persistent_workers=(_nw > 0))
    model.to(DEVICE)
    torch.backends.cudnn.benchmark = True
    optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG["lr"],
                                  weight_decay=CONFIG.get("weight_decay", 1e-4))
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max",
        factor=CONFIG.get("sched_factor", 0.5),
        patience=CONFIG.get("sched_patience", 10),
        min_lr=CONFIG.get("min_lr", 1e-6))
    use_amp = CONFIG.get("use_amp", True) and DEVICE.type == "cuda"
    scaler  = torch.amp.GradScaler("cuda", enabled=use_amp)
    if CONFIG.get("class_weights") is not None:
        cw = torch.tensor(CONFIG["class_weights"], dtype=torch.float32, device=DEVICE)
        loss_fn = nn.CrossEntropyLoss(weight=cw)
        log.info(f"Using class weights: {CONFIG['class_weights']}")
    else:
        loss_fn = nn.CrossEntropyLoss()
    start_epoch = 1
    best_miou   = -1.0
    no_improve  = 0
    history     = []
    auto_reg_no_improve = 0
    auto_reg_triggers   = 0
    resume_epoch, resume_path = latest_periodic()
    if resume_path:
        log.info(f"Resuming from {resume_path} (epoch {resume_epoch})")
        ckpt = torch.load(resume_path, map_location="cpu", weights_only=False)
        model.load_state_dict(ckpt["model_state"])
        optimizer.load_state_dict(ckpt["optimizer_state"])
        if "scheduler_state" in ckpt:
            scheduler.load_state_dict(ckpt["scheduler_state"])
        else:
            log.warning("Checkpoint has no scheduler_state (older checkpoint?) — "
                       "the LR-plateau counter is starting fresh from this resume, "
                       "so LR may not reduce as promptly as it should have.")
        for state in optimizer.state.values():
            for k, v in state.items():
                if isinstance(v, torch.Tensor):
                    state[k] = v.to(DEVICE)
        start_epoch = ckpt["epoch"] + 1
        best_miou   = ckpt.get("best_miou",   -1.0)
        no_improve  = ckpt.get("no_improve",   0)
        history     = ckpt.get("history",      [])
        auto_reg_no_improve = ckpt.get("auto_reg_no_improve", 0)
        auto_reg_triggers   = ckpt.get("auto_reg_triggers", 0)
        if auto_reg_triggers > 0:
            wd = min(CONFIG.get("weight_decay", 1e-4) *
                     (CONFIG.get("auto_reg_wd_factor", 2.0) ** auto_reg_triggers),
                     CONFIG.get("auto_reg_wd_max", 1e-2))
            for g in optimizer.param_groups:
                g["weight_decay"] = wd
            dp_add = CONFIG.get("auto_reg_dropout_step", 0.05) * auto_reg_triggers
            for m in model.modules():
                if isinstance(m, nn.Dropout):
                    m.p = min(m.p + dp_add, CONFIG.get("auto_reg_dropout_max", 0.6))
            log.info(f"Restored auto-regularization state: {auto_reg_triggers} "
                     f"prior trigger(s), weight_decay={wd:.5f}")
        log.info(f"Resumed: start_epoch={start_epoch}  best_miou={best_miou:.4f}")
    else:
        log.info(f"No checkpoint found — training from scratch.")
    if start_epoch > CONFIG["epochs"]:
        log.info("Training already complete. Loading best model.")
        state = torch.load(best_path, map_location="cpu", weights_only=False)
        model.load_state_dict(state["model_state"])
        return model
    run_name = f"{model_name}_seg"
    with (mlflow.start_run(run_name=run_name) if MLFLOW_OK
          else open(os.devnull, "w")) as _:
        if MLFLOW_OK:
            global LAST_RUN_ID
            LAST_RUN_ID = mlflow.active_run().info.run_id
            mlflow.log_params({
                "model"           : model_name,
                "dataset"         : os.path.basename(os.path.normpath(
                                     CONFIG.get("local_data_dir", "unknown_dataset")))
                                     if "local_data_dir" in CONFIG else "wood_powder_lidar",
                "n_train_files"   : len(TRAIN_FILES),
                "n_val_files"     : len(VAL_FILES),
                "n_test_files"    : len(TEST_FILES),
                "in_channels"     : CONFIG["in_channels"],
                "num_classes"     : NUM_CLASSES,
                "epochs"          : CONFIG["epochs"],
                "patience"        : CONFIG["patience"],
                "batch_size"      : CONFIG["batch_size"],
                "num_points"      : CONFIG["num_points"],
                "lr"              : CONFIG["lr"],
                "cv_folds"        : CONFIG.get("cv_folds", 1),
                "auto_regularize" : CONFIG.get("auto_regularize", False),
                "start_epoch"     : start_epoch,
            })
        accumulation_steps = CONFIG.get("grad_accum", 4)
        for epoch in range(start_epoch, CONFIG["epochs"] + 1):
            model.train()
            optimizer.zero_grad()
            train_loss_sum = 0.0
            train_correct = 0
            train_total = 0
            for i, (x, y) in enumerate(train_loader):
                x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
                with torch.amp.autocast("cuda", enabled=use_amp):
                    out  = model(x)
                    loss = loss_fn(out, y)
                train_loss_sum += loss.item() * x.size(0)
                preds = out.argmax(dim=1)
                train_correct += (preds == y).sum().item()
                train_total += y.numel()
                scaler.scale(loss / accumulation_steps).backward()
                if (i + 1) % accumulation_steps == 0 or (i + 1) == len(train_loader):
                    scaler.step(optimizer)
                    scaler.update()
                    optimizer.zero_grad()
            epoch_train_loss = train_loss_sum / len(train_loader.dataset)
            epoch_train_acc = train_correct / train_total
            model.eval()
            val_loss_sum = 0.0
            val_correct = 0
            val_total = 0
            all_true, all_pred = [], []
            with torch.no_grad():
                for x, y in val_loader:
                    x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
                    with torch.amp.autocast("cuda", enabled=use_amp):
                        out  = model(x)
                        loss = loss_fn(out, y)
                    val_loss_sum += loss.item() * x.size(0)
                    preds = out.argmax(dim=1)
                    val_correct += (preds == y).sum().item()
                    val_total += y.numel()
                    all_pred.extend(preds.cpu().numpy().ravel())
                    all_true.extend(y.cpu().numpy().ravel())
            epoch_val_loss = val_loss_sum / len(val_loader.dataset)
            epoch_val_acc = val_correct / val_total
            miou = compute_miou(np.array(all_true), np.array(all_pred), NUM_CLASSES)
            dice = compute_dice(np.array(all_true), np.array(all_pred), NUM_CLASSES)
            history.append({
                "epoch": epoch, "train_loss": epoch_train_loss, "val_loss": epoch_val_loss,
                "train_acc": epoch_train_acc, "val_acc": epoch_val_acc,
                "val_miou": miou, "val_dice": dice,
                "lr": optimizer.param_groups[0]["lr"],
            })
            log.info(
                f"Epoch {epoch:3d}/{CONFIG['epochs']} | "
                f"Train Loss: {epoch_train_loss:.4f} | Val Loss: {epoch_val_loss:.4f} | "
                f"Train Acc: {epoch_train_acc:.4f} | Val Acc: {epoch_val_acc:.4f} | "
                f"mIoU: {miou:.4f} | Dice: {dice:.4f}"
            )
            if MLFLOW_OK:
                _epoch_metrics = {
                    "train_loss": epoch_train_loss,
                    "val_loss": epoch_val_loss,
                    "train_acc": epoch_train_acc,
                    "val_acc": epoch_val_acc,
                    "val_miou": miou,
                    "val_dice": dice,
                    "lr": optimizer.param_groups[0]["lr"],
                }
                mlflow.log_metrics(_epoch_metrics, step=epoch)
                with mlflow.start_run(run_name=f"epoch_{epoch:04d}", nested=True):
                    mlflow.log_metrics(_epoch_metrics, step=epoch)
            scheduler.step(miou)
            if miou > best_miou + 1e-4:
                best_miou  = miou
                no_improve = 0
                torch.save({"model_state": model.state_dict(),
                            "best_val_miou": best_miou,
                            "in_channels": CONFIG["in_channels"],
                            "num_classes": NUM_CLASSES,
                            "num_points": CONFIG["num_points"],
                            "use_linearity"   : CONFIG["use_linearity"],
                            "use_planarity"   : CONFIG["use_planarity"],
                            "use_sphericity"  : CONFIG["use_sphericity"],
                            "use_verticality" : CONFIG.get("use_verticality", False),
                            "use_eigenentropy": CONFIG.get("use_eigenentropy", False),
                            "use_rgb"         : CONFIG["use_rgb"],
                            "model_hparams"   : {k: CONFIG[k] for k in CONFIG
                                                if k.startswith(("kp_", "rl_", "sp_", "pn2_", "pnx_"))},
                            "auto_reg_no_improve": auto_reg_no_improve,
                            "auto_reg_triggers"  : auto_reg_triggers}, best_path)
                log.info(f"  ✓ New best mIoU {best_miou:.4f} → saved {best_path}")
                auto_reg_no_improve = 0
            else:
                no_improve += 1
                auto_reg_no_improve += 1

            if (CONFIG.get("auto_regularize", False)
                    and auto_reg_no_improve >= CONFIG.get("auto_reg_patience", 15)
                    and auto_reg_triggers < CONFIG.get("auto_reg_max_triggers", 3)):
                auto_reg_triggers += 1
                new_wd = min(optimizer.param_groups[0]["weight_decay"]
                             * CONFIG.get("auto_reg_wd_factor", 2.0),
                             CONFIG.get("auto_reg_wd_max", 1e-2))
                for g in optimizer.param_groups:
                    g["weight_decay"] = new_wd
                n_dropout_bumped = 0
                for m in model.modules():
                    if isinstance(m, nn.Dropout):
                        m.p = min(m.p + CONFIG.get("auto_reg_dropout_step", 0.05),
                                  CONFIG.get("auto_reg_dropout_max", 0.6))
                        n_dropout_bumped += 1
                auto_reg_no_improve = 0
                log.info(f"  ⚠ Auto-regularize trigger #{auto_reg_triggers}: val mIoU "
                         f"stalled {CONFIG.get('auto_reg_patience', 15)} epochs — "
                         f"weight_decay→{new_wd:.5f}, bumped {n_dropout_bumped} "
                         f"dropout layer(s) by "
                         f"{CONFIG.get('auto_reg_dropout_step', 0.05)}")

            if epoch % SAVE_EVERY == 0 or epoch == CONFIG["epochs"]:
                periodic_path = os.path.join(ckpt_dir,
                                             f"{model_name}_epoch_{epoch:04d}.pth")
                torch.save({
                    "model_state"     : model.state_dict(),
                    "optimizer_state" : optimizer.state_dict(),
                    "scheduler_state" : scheduler.state_dict(),
                    "epoch"           : epoch,
                    "best_miou"       : best_miou,
                    "no_improve"      : no_improve,
                    "history"         : history,
                    "in_channels"     : CONFIG["in_channels"],
                    "num_classes"     : NUM_CLASSES,
                    "num_points"      : CONFIG["num_points"],
                    "use_linearity"   : CONFIG["use_linearity"],
                    "use_planarity"   : CONFIG["use_planarity"],
                    "use_sphericity"  : CONFIG["use_sphericity"],
                    "use_verticality" : CONFIG.get("use_verticality", False),
                    "use_eigenentropy": CONFIG.get("use_eigenentropy", False),
                    "use_rgb"         : CONFIG["use_rgb"],
                    "model_hparams"   : {k: CONFIG[k] for k in CONFIG
                                        if k.startswith(("kp_", "rl_", "sp_", "pn2_", "pnx_"))},
                    "auto_reg_no_improve": auto_reg_no_improve,
                    "auto_reg_triggers"  : auto_reg_triggers,
                }, periodic_path)
                log.info(f"  💾 Periodic checkpoint saved → {periodic_path}")
                prune_old_checkpoints(keep=2)
            if no_improve >= CONFIG["patience"]:
                log.info(f"Early stop at epoch {epoch} "
                         f"(no improvement for {CONFIG['patience']} epochs)")
                break
        if MLFLOW_OK:
            mlflow.log_metric("best_val_miou", best_miou)
            try:
                import matplotlib
                matplotlib.use("Agg")
                import matplotlib.pyplot as _plt
                epochs_h = [h["epoch"] for h in history]
                fig, axes = _plt.subplots(2, 2, figsize=(14, 10))
                fig.suptitle(f"{model_name} — Training Diagnostics "
                            f"(overfitting shows as train/val curves diverging)",
                            fontsize=13)

                ax = axes[0, 0]
                ax.plot(epochs_h, [h["train_loss"] for h in history], label="train_loss")
                ax.plot(epochs_h, [h["val_loss"] for h in history], label="val_loss")
                ax.set_xlabel("epoch"); ax.set_ylabel("loss")
                ax.set_title("Loss"); ax.legend(); ax.grid(alpha=0.3)

                ax = axes[0, 1]
                ax.plot(epochs_h, [h["train_acc"] for h in history], label="train_acc")
                ax.plot(epochs_h, [h["val_acc"] for h in history], label="val_acc")
                ax.set_xlabel("epoch"); ax.set_ylabel("accuracy")
                ax.set_title("Accuracy"); ax.legend(); ax.grid(alpha=0.3)

                ax = axes[1, 0]
                ax.plot(epochs_h, [h["val_miou"] for h in history],
                       label="val_mIoU", color="green")
                ax.plot(epochs_h, [h["val_dice"] for h in history],
                       label="val_Dice", color="orange")
                ax.axhline(best_miou, color="red", linestyle="--",
                          label=f"best_mIoU={best_miou:.4f}")
                ax.set_xlabel("epoch"); ax.set_ylabel("score")
                ax.set_title("mIoU / Dice"); ax.legend(); ax.grid(alpha=0.3)

                ax = axes[1, 1]
                ax.plot(epochs_h, [h["lr"] for h in history], color="purple")
                ax.set_xlabel("epoch"); ax.set_ylabel("learning rate")
                ax.set_yscale("log"); ax.set_title("LR schedule")
                ax.grid(alpha=0.3)

                _plt.tight_layout()
                _diag_path = os.path.join(ckpt_dir, f"{model_name}_diagnostics.png")
                _plt.savefig(_diag_path, dpi=120)
                _plt.close(fig)
                mlflow.log_artifact(_diag_path)
            except Exception as _e:
                log.warning(f"Could not generate/log diagnostic plot: {_e}")
    if os.path.exists(best_path):
        state = torch.load(best_path, map_location="cpu", weights_only=False)
        model.load_state_dict(state["model_state"])
        log.info(f"Loaded best model (mIoU={best_miou:.4f}) from {best_path}")
    model.to("cpu")
    return model

In [13]:
if CV_FOLDS is None:
    model = train_model(model, MODEL_NAME)
else:
    _base_ckpt_dir = CONFIG["checkpoint_dir"]
    fold_mious = []
    for fold_idx, (TRAIN_FILES, VAL_FILES) in enumerate(CV_FOLDS):
        log.info(f"\n{'='*60}\n=== FOLD {fold_idx + 1}/{len(CV_FOLDS)} ===\n{'='*60}")
        CONFIG["checkpoint_dir"] = f"{_base_ckpt_dir}_fold{fold_idx}"
        os.makedirs(CONFIG["checkpoint_dir"], exist_ok=True)
        model = build_model()
        model = train_model(model, MODEL_NAME)
        best_path = os.path.join(CONFIG["checkpoint_dir"], f"{MODEL_NAME}_best.pth")
        ckpt = torch.load(best_path, map_location="cpu", weights_only=False)
        fold_miou = ckpt.get("best_val_miou", float("nan"))
        fold_mious.append(fold_miou)
        log.info(f"Fold {fold_idx + 1} best val mIoU: {fold_miou:.4f}")
    CONFIG["checkpoint_dir"] = _base_ckpt_dir
    fold_mious = np.array(fold_mious)
    log.info(f"\n{'='*60}")
    log.info(f"K-FOLD CV RESULTS ({CONFIG['cv_folds']} folds)")
    log.info(f"  per-fold mIoU: {[f'{m:.4f}' for m in fold_mious]}")
    log.info(f"  mean ± std   : {fold_mious.mean():.4f} ± {fold_mious.std():.4f}")
    log.info(f"{'='*60}")
    best_fold = int(np.argmax(fold_mious))
    best_fold_path = os.path.join(f"{_base_ckpt_dir}_fold{best_fold}",
                                  f"{MODEL_NAME}_best.pth")
    state = torch.load(best_fold_path, map_location="cpu", weights_only=False)
    model = build_model()
    model.load_state_dict(state["model_state"])
    log.info(f"Loaded fold {best_fold + 1}'s weights (best mIoU="
             f"{fold_mious[best_fold]:.4f}) into `model` for downstream use")

2026-07-20 22:04:15,173 | INFO | Using class weights: [1.2528783535934613, 0.7471216464065389]
2026-07-20 22:04:15,174 | INFO | No checkpoint found — training from scratch.
2026-07-20 22:09:38,057 | INFO | Epoch   1/15 | Train Loss: 0.4213 | Val Loss: 0.2992 | Train Acc: 0.8113 | Val Acc: 0.8677 | mIoU: 0.7385 | Dice: 0.8468
2026-07-20 22:09:38,310 | INFO |   ✓ New best mIoU 0.7385 → saved ../checkpoints/PointCloudDataset_SlidingWindow_10_channel_v2/PointTransformerV2_best.pth


🏃 View run epoch_0001 at: http://localhost:5000/#/experiments/13/runs/0c7f5f766f0c4cce8484cecc315c569e
🧪 View experiment at: http://localhost:5000/#/experiments/13


2026-07-20 22:15:12,821 | INFO | Epoch   2/15 | Train Loss: 0.2964 | Val Loss: 0.2660 | Train Acc: 0.8866 | Val Acc: 0.9021 | mIoU: 0.7791 | Dice: 0.8725
2026-07-20 22:15:13,136 | INFO |   ✓ New best mIoU 0.7791 → saved ../checkpoints/PointCloudDataset_SlidingWindow_10_channel_v2/PointTransformerV2_best.pth


🏃 View run epoch_0002 at: http://localhost:5000/#/experiments/13/runs/ec9e41992caa481d95c8c776a5c3d9c1
🧪 View experiment at: http://localhost:5000/#/experiments/13


2026-07-20 22:20:48,649 | INFO | Epoch   3/15 | Train Loss: 0.2677 | Val Loss: 0.3001 | Train Acc: 0.8994 | Val Acc: 0.8674 | mIoU: 0.7419 | Dice: 0.8495


...

🏃 View run epoch_0014 at: http://localhost:5000/#/experiments/13/runs/cf8e495b7e314d5698f7b0d4483c7343
🧪 View experiment at: http://localhost:5000/#/experiments/13


2026-07-20 23:28:09,728 | INFO | Epoch  15/15 | Train Loss: 0.1955 | Val Loss: 0.2130 | Train Acc: 0.9353 | Val Acc: 0.9220 | mIoU: 0.8305 | Dice: 0.9061
2026-07-20 23:28:10,057 | INFO |   💾 Periodic checkpoint saved → ../checkpoints/PointCloudDataset_SlidingWindow_10_channel_v2/PointTransformerV2_epoch_0015.pth


🏃 View run epoch_0015 at: http://localhost:5000/#/experiments/13/runs/7a275b39083c40f482d6babf56c6e63c
🧪 View experiment at: http://localhost:5000/#/experiments/13


2026-07-20 23:28:10,707 | INFO | Loaded best model (mIoU=0.8570) from ../checkpoints/PointCloudDataset_SlidingWindow_10_channel_v2/PointTransformerV2_best.pth


🏃 View run PointTransformerV2_seg at: http://localhost:5000/#/experiments/13/runs/121508ef12194a32af68c4851eac59b9
🧪 View experiment at: http://localhost:5000/#/experiments/13


## Visualization helper

In [14]:
def visualize_segmentation(points, predictions, title="Segmentation"):
    colors = np.zeros((len(points), 3), dtype=np.float64)
    colors[predictions == CONFIG["target_class"]] = [0.0, 0.8, 0.0]
    colors[predictions != CONFIG["target_class"]] = [0.8, 0.0, 0.0]
    pcd = o3d.geometry.PointCloud()
    center = points.mean(axis=0)
    pcd.points = o3d.utility.Vector3dVector(
        (points - center).astype(np.float64))
    pcd.colors = o3d.utility.Vector3dVector(colors)
    span = float((points.max(0) - points.min(0)).max()) * 0.5
    axes = o3d.geometry.TriangleMesh.create_coordinate_frame(size=span)
    n_target = int((predictions == CONFIG["target_class"]).sum())
    n_other  = len(predictions) - n_target
    full_title = (f"{title}  |  green(target)={n_target:,}  "
                  f"red(others)={n_other:,}")
    o3d.visualization.draw_geometries([pcd, axes],
                                       window_name=full_title,
                                       width=1280, height=800)

## Inference helper

In [15]:
from scipy.spatial import cKDTree as _InferKDTree
@torch.no_grad()
def predict_full_cloud(model, path, overlap_factor=2):
    model.eval()
    pts, feat, lbl = get_features(path)
    n = len(feat)
    N = CONFIG["num_points"]
    B = CONFIG["batch_size"]
    log.info(f"predict_full_cloud: {n:,} points, N={N:,}/chunk, batch={B}, "
             f"device={DEVICE} — this can take a while on CPU, progress below")
    model.to(DEVICE)
    if n <= N:
        pad_idx = np.random.RandomState(0).choice(n, N, replace=True)
        x   = torch.from_numpy(feat[pad_idx][None].transpose(0, 2, 1)).to(DEVICE)
        out = model(x).argmax(dim=1).cpu().numpy().ravel()
        preds = np.zeros(n, np.int64)
        preds[pad_idx] = out
        model.to("cpu")
        return pts, preds, lbl
    tree      = _InferKDTree(feat[:, :3])
    covered   = np.zeros(n, bool)
    vote_sum  = np.zeros((n, NUM_CLASSES), np.int32)
    rng       = np.random.RandomState(0)
    batch_ids = []
    flush_count = 0
    def flush():
        nonlocal flush_count
        if not batch_ids:
            return
        x   = torch.from_numpy(
                  feat[np.stack(batch_ids)].transpose(0, 2, 1)).to(DEVICE)
        out = model(x).argmax(dim=1).cpu().numpy()
        for ids, o in zip(batch_ids, out):
            np.add.at(vote_sum, (ids, o), 1)
        batch_ids.clear()
        flush_count += 1
        pct = covered.mean() * 100
        log.info(f"  flush {flush_count}: {covered.sum():,}/{n:,} pts "
                 f"covered ({pct:.1f}%)")
        import sys; sys.stdout.flush()
    target_visits = max(1, overlap_factor)
    visit_count = np.zeros(n, np.int32)
    max_iters = int(np.ceil(n / N * target_visits)) + 4
    for _ in range(max_iters):
        if (visit_count >= target_visits).all():
            break
        least = np.flatnonzero(visit_count == visit_count.min())
        seed  = least[rng.randint(len(least))]
        _, idx = tree.query(pts[seed], k=N)
        idx = np.atleast_1d(idx)
        batch_ids.append(idx)
        visit_count[idx] += 1
        covered[idx] = True
        if len(batch_ids) == B:
            flush()
    flush()
    if not covered.all() and covered.any():
        missing     = np.flatnonzero(~covered)
        covered_idx = np.flatnonzero(covered)
        _, nn_local = _InferKDTree(pts[covered_idx]).query(pts[missing], k=1)
        vote_sum[missing] = vote_sum[covered_idx[nn_local]]
    preds = vote_sum.argmax(axis=1).astype(np.int64)
    model.to("cpu")
    return pts, preds, lbl

## Per-file Train/Val Report (log only, no saving)

In [16]:
log.info("=== TRAIN/VAL PER-FILE REPORT ===")
trval_names, trval_true, trval_pred = [], [], []
for path in TRAIN_FILES:
    name = os.path.splitext(os.path.basename(path))[0]
    pts, preds, lbl = predict_full_cloud(model, path)
    trval_names.append(f"{name} [train]"); trval_true.append(lbl); trval_pred.append(preds)
per_file_report(trval_names, trval_true, trval_pred, tag="TRAIN",
                target_class=CONFIG["target_class"])
val_names, val_true, val_pred = [], [], []
for path in VAL_FILES:
    name = os.path.splitext(os.path.basename(path))[0]
    pts, preds, lbl = predict_full_cloud(model, path)
    val_names.append(f"{name} [val]"); val_true.append(lbl); val_pred.append(preds)
per_file_report(val_names, val_true, val_pred, tag="VAL",
                target_class=CONFIG["target_class"])

2026-07-20 23:28:10,946 | INFO | === TRAIN/VAL PER-FILE REPORT ===
2026-07-20 23:28:10,947 | INFO | predict_full_cloud: 296,835 points, N=16,384/chunk, batch=8, device=cuda — this can take a while on CPU, progress below
2026-07-20 23:28:12,314 | INFO |   flush 1: 16,384/296,835 pts covered (5.5%)
2026-07-20 23:28:13,594 | INFO |   flush 2: 16,384/296,835 pts covered (5.5%)
2026-07-20 23:28:14,892 | INFO |   flush 3: 16,384/296,835 pts covered (5.5%)
2026-07-20 23:28:16,209 | INFO |   flush 4: 16,384/296,835 pts covered (5.5%)
2026-07-20 23:28:17,516 | INFO |   flush 5: 16,384/296,835 pts covered (5.5%)
2026-07-20 23:28:17,701 | INFO |   flush 6: 16,384/296,835 pts covered (5.5%)
2026-07-20 23:28:22,716 | INFO | predict_full_cloud: 450,029 points, N=16,384/chunk, batch=8, device=cuda — this can take a while on CPU, progress below
2026-07-20 23:28:24,325 | INFO |   flush 1: 16,384/450,029 pts covered (3.6%)
2026-07-20 23:28:25,901 | INFO |   flush 2: 16,384/450,029 pts covered (3.6%)
202


PER-FILE SEGMENTATION REPORT — TRAIN
File                      IoU   Dice   Prec    Rec       TP       FP       FN       TN   GT_Wood  Pred_Wood
-----------------------------------------------------------------------------------------------------------
sample_data_0039 [train]  0.456  0.626  0.773  0.526  111,149   32,660  100,061   52,965   211,210    143,809
sample_data_0027 [train]  0.439  0.610  0.774  0.503  129,128   37,633  127,385  155,883   256,513    166,761
sample_data_0025 [train]  0.305  0.468  0.850  0.322   56,375    9,969  118,434   97,285   174,809     66,344
sample_data_0021 [train]  0.123  0.218  0.409  0.149   28,728   41,530  164,216  120,800   192,944     70,258
sample_data_007 [train]  0.561  0.719  0.727  0.710  188,793   70,870   77,022   59,373   265,815    259,663
sample_data_0023 [train]  0.552  0.712  0.552  1.000  258,972  209,996        0        0   258,972    468,968
sample_data_0018 [train]  0.265  0.419  0.677  0.304   78,925   37,723  181,060  147,08

2026-07-20 23:36:13,619 | INFO |   flush 1: 16,384/311,676 pts covered (5.3%)
2026-07-20 23:36:15,297 | INFO |   flush 2: 16,384/311,676 pts covered (5.3%)
2026-07-20 23:36:16,988 | INFO |   flush 3: 16,384/311,676 pts covered (5.3%)
2026-07-20 23:36:18,693 | INFO |   flush 4: 16,384/311,676 pts covered (5.3%)
2026-07-20 23:36:20,377 | INFO |   flush 5: 16,384/311,676 pts covered (5.3%)
2026-07-20 23:36:21,141 | INFO |   flush 6: 16,384/311,676 pts covered (5.3%)
2026-07-20 23:36:25,622 | INFO | predict_full_cloud: 420,576 points, N=16,384/chunk, batch=8, device=cuda — this can take a while on CPU, progress below
2026-07-20 23:36:27,338 | INFO |   flush 1: 16,384/420,576 pts covered (3.9%)
2026-07-20 23:36:28,978 | INFO |   flush 2: 16,384/420,576 pts covered (3.9%)
2026-07-20 23:36:30,598 | INFO |   flush 3: 16,384/420,576 pts covered (3.9%)
2026-07-20 23:36:32,271 | INFO |   flush 4: 16,384/420,576 pts covered (3.9%)
2026-07-20 23:36:33,927 | INFO |   flush 5: 16,384/420,576 pts cove


PER-FILE SEGMENTATION REPORT — VAL
File                      IoU   Dice   Prec    Rec       TP       FP       FN       TN   GT_Wood  Pred_Wood
-----------------------------------------------------------------------------------------------------------
sample_data_0013 [val]  0.289  0.448  0.685  0.333   69,369   31,968  138,998   71,341   208,367    101,337
sample_data_0020 [val]  0.293  0.453  0.862  0.307   87,464   13,984  197,279  121,849   284,743    101,448
sample_data_0017 [val]  0.128  0.226  0.913  0.129   50,018    4,788  337,106  164,487   387,124     54,806
sample_data_0012 [val]  0.650  0.788  0.707  0.891  214,379   89,008   26,221   70,597   240,600    303,387
sample_data_0015 [val]  0.063  0.118  0.921  0.063   17,549    1,511  261,065  111,243   278,614     19,060
sample_data_0031 [val]  0.792  0.884  0.960  0.818  168,361    6,990   37,343   23,645   205,704    175,351
----------------------------------------------------------------------------------------------------

In [17]:
from scipy.spatial import Delaunay, cKDTree
def sor_filter(pts, k=16, std_ratio=2.0):
    pts = np.asarray(pts, np.float64)
    if len(pts) <= k + 1:
        return pts, np.ones(len(pts), bool)
    tree = cKDTree(pts)
    d, _ = tree.query(pts, k=k + 1)
    mean_dist = d[:, 1:].mean(axis=1)
    thr  = mean_dist.mean() + std_ratio * mean_dist.std()
    mask = mean_dist <= thr
    return pts[mask], mask
def extract_main_cluster(pts, eps=0.15, min_samples=8):
    pts = np.asarray(pts, np.float64)
    if len(pts) < min_samples * 2:
        return pts
    try:
        pcd = o3d.geometry.PointCloud()
        pcd.points = o3d.utility.Vector3dVector(pts)
        labels = np.asarray(pcd.cluster_dbscan(eps=eps, min_points=min_samples))
        valid = labels[labels >= 0]
        if valid.size == 0:
            return pts
        main_lbl = np.bincount(valid).argmax()
        result = pts[labels == main_lbl]
        return result if len(result) >= min_samples else pts
    except Exception as e:
        log.warning(f"[TIN] cluster extraction failed ({e}) -> using all points")
        return pts
def tin_integrate(pts, floor_pct=0.5):
    pts = np.asarray(pts, np.float64)
    if len(pts) < 4:
        return float("nan")
    floor_z = np.percentile(pts[:, 2], floor_pct)
    h  = pts[:, 2] - floor_z
    xy = pts[:, :2]
    try:
        tri = Delaunay(xy)
    except Exception as e:
        log.warning(f"[TIN] Delaunay failed ({e})")
        return float("nan")
    t = tri.simplices
    p0, p1, p2 = xy[t[:, 0]], xy[t[:, 1]], xy[t[:, 2]]
    h0, h1, h2 = h[t[:, 0]], h[t[:, 1]], h[t[:, 2]]
    cross  = ((p1[:, 0] - p0[:, 0]) * (p2[:, 1] - p0[:, 1])
             - (p1[:, 1] - p0[:, 1]) * (p2[:, 0] - p0[:, 0]))
    area2d = np.abs(cross) / 2.0
    mean_h = (h0 + h1 + h2) / 3.0
    valid  = mean_h > 0
    return float(np.sum(area2d[valid] * mean_h[valid]))
def bootstrap_tin_confidence(pts, n_bootstrap=80, subsample_frac=0.85,
                             floor_pct=0.5, seed=0):
    rng = np.random.RandomState(seed)
    n = len(pts)
    n_sub = max(10, int(n * subsample_frac))
    estimates = []
    for _ in range(n_bootstrap):
        idx = rng.choice(n, n_sub, replace=False)
        v = tin_integrate(pts[idx], floor_pct=floor_pct)
        if np.isfinite(v) and v > 0:
            estimates.append(v)
    if len(estimates) < 5:
        return None
    arr = np.array(estimates)
    return {"std":     round(float(arr.std()), 5),
            "ci_low":  round(float(np.percentile(arr, 2.5)), 5),
            "ci_high": round(float(np.percentile(arr, 97.5)), 5)}
def vol_TIN(pts, sor_k=16, sor_std=2.0, cluster_eps=0.15,
            floor_pct=0.5, run_bootstrap=True, n_bootstrap=80):
    pts = np.asarray(pts, np.float64)
    if len(pts) < 10:
        return float("nan"), None
    pts, _ = sor_filter(pts, k=sor_k, std_ratio=sor_std)
    if len(pts) < 10:
        return float("nan"), None
    pts = extract_main_cluster(pts, eps=cluster_eps)
    if len(pts) < 10:
        return float("nan"), None
    volume = tin_integrate(pts, floor_pct=floor_pct)
    conf = None
    if run_bootstrap and len(pts) >= 20:
        conf = bootstrap_tin_confidence(pts, n_bootstrap=n_bootstrap,
                                        floor_pct=floor_pct)
    return float(volume), conf

## Testing  (held-out test files with labels)

In [18]:
log.info("=== TESTING ===")
test_names, test_true, test_pred = [], [], []
for path in TEST_FILES:
    name = os.path.splitext(os.path.basename(path))[0]
    pts, preds, lbl = predict_full_cloud(model, path)
    test_names.append(name); test_true.append(lbl); test_pred.append(preds)
per_file_report(test_names, test_true, test_pred, tag="TEST",
                target_class=CONFIG["target_class"])

if MLFLOW_OK and test_true and all(t is not None for t in test_true):
    from sklearn.metrics import classification_report
    _all_true = np.concatenate(test_true)
    _all_pred = np.concatenate(test_pred)
    _report_dict = classification_report(
        _all_true, _all_pred, target_names=["background", "wood_powder"],
        output_dict=True, zero_division=0)
    _report_text = classification_report(
        _all_true, _all_pred, target_names=["background", "wood_powder"],
        zero_division=0)
    log.info("=== TEST CLASSIFICATION REPORT ===\n" + _report_text)

    _test_miou = compute_miou(_all_true, _all_pred, NUM_CLASSES)
    _test_dice = compute_dice(_all_true, _all_pred, NUM_CLASSES)

    try:
        with mlflow.start_run(run_id=LAST_RUN_ID):
            mlflow.log_metrics({
                "test_accuracy"          : _report_dict["accuracy"],
                "test_miou"              : _test_miou,
                "test_dice"              : _test_dice,
                "test_precision_target"  : _report_dict["wood_powder"]["precision"],
                "test_recall_target"     : _report_dict["wood_powder"]["recall"],
                "test_f1_target"         : _report_dict["wood_powder"]["f1-score"],
                "test_precision_bg"      : _report_dict["background"]["precision"],
                "test_recall_bg"         : _report_dict["background"]["recall"],
                "test_f1_bg"             : _report_dict["background"]["f1-score"],
            })
            _report_path = os.path.join(CONFIG["checkpoint_dir"],
                                        f"{MODEL_NAME}_test_classification_report.txt")
            with open(_report_path, "w") as _f:
                _f.write(_report_text)
            mlflow.log_artifact(_report_path)
    except Exception as _e:
        log.warning(f"Could not log test metrics to MLflow (run_id={LAST_RUN_ID}): {_e}")
for path in TEST_FILES[:CONFIG["max_vis_files"]]:
    name = os.path.splitext(os.path.basename(path))[0]
    pts, preds, _ = predict_full_cloud(model, path)
    target_pts   = pts[preds == CONFIG["target_class"]]
    volume, conf = vol_TIN(target_pts)
    n_tot = len(pts)
    n_tgt = len(target_pts)
    print(f"\n{'─'*55}")
    print(f"  File          : {name}")
    print(f"  Total Points  : {n_tot:,}")
    print(f"  Target Points : {n_tgt:,}")
    print(f"  Other Points  : {n_tot - n_tgt:,}")
    print(f"  Target Ratio  : {n_tgt / max(n_tot, 1) * 100:.2f}%")
    print(f"  Est. Volume   : {volume:.4f} m³")
    if conf:
        print(f"  95% CI        : [{conf['ci_low']:.4f} – {conf['ci_high']:.4f}] m³"
              f"  (±{conf['std']:.4f})")
    print(f"{'─'*55}")
    visualize_segmentation(pts, preds,
                           title=f"{MODEL_NAME} | {name} | V={volume:.4f} m³")

2026-07-20 23:37:25,984 | INFO | === TESTING ===
2026-07-20 23:37:27,147 | INFO |   SOR: removed 5160 outlier pts (sample_data_0045.las)
2026-07-20 23:37:27,650 | INFO |   Cluster-filter: removed 1683 pts in small floating clusters (sample_data_0045.las)
2026-07-20 23:37:36,888 | INFO | predict_full_cloud: 416,144 points, N=16,384/chunk, batch=8, device=cuda — this can take a while on CPU, progress below
2026-07-20 23:37:38,330 | INFO |   flush 1: 16,384/416,144 pts covered (3.9%)
2026-07-20 23:37:39,690 | INFO |   flush 2: 16,384/416,144 pts covered (3.9%)
2026-07-20 23:37:41,058 | INFO |   flush 3: 16,384/416,144 pts covered (3.9%)
2026-07-20 23:37:42,465 | INFO |   flush 4: 16,384/416,144 pts covered (3.9%)
2026-07-20 23:37:43,847 | INFO |   flush 5: 16,384/416,144 pts covered (3.9%)
2026-07-20 23:37:45,221 | INFO |   flush 6: 16,384/416,144 pts covered (3.9%)
2026-07-20 23:37:46,420 | INFO |   flush 7: 16,384/416,144 pts covered (3.9%)
2026-07-20 23:37:49,271 | INFO |   SOR: remove


PER-FILE SEGMENTATION REPORT — TEST
File                      IoU   Dice   Prec    Rec       TP       FP       FN       TN   GT_Wood  Pred_Wood
-----------------------------------------------------------------------------------------------------------
sample_data_0045        0.286  0.445  0.876  0.298   66,147    9,340  155,782  184,875   221,929     75,487
sample_data_0032        0.190  0.319  0.852  0.196   39,069    6,792  160,137  174,618   199,206     45,861
sample_data_0033        0.107  0.193  0.333  0.136   29,324   58,812  186,395  172,753   215,719     88,136
sample_data_008         0.203  0.337  0.917  0.206   58,400    5,283  224,545  155,904   282,945     63,683
sample_data_0041        0.254  0.405  0.836  0.267   59,243   11,594  162,620  132,644   221,863     70,837
sample_data_006         0.305  0.468  0.307  0.987  190,086  429,576    2,577   18,631   192,663    619,662
---------------------------------------------------------------------------------------------------

2026-07-20 23:40:00,420 | INFO | === TEST CLASSIFICATION REPORT ===
              precision    recall  f1-score   support

  background       0.48      0.62      0.54   1360822
 wood_powder       0.46      0.33      0.38   1334325

    accuracy                           0.48   2695147
   macro avg       0.47      0.47      0.46   2695147
weighted avg       0.47      0.48      0.46   2695147

2026-07-20 23:40:00,782 | INFO | predict_full_cloud: 416,144 points, N=16,384/chunk, batch=8, device=cuda — this can take a while on CPU, progress below


🏃 View run PointTransformerV2_seg at: http://localhost:5000/#/experiments/13/runs/121508ef12194a32af68c4851eac59b9
🧪 View experiment at: http://localhost:5000/#/experiments/13


2026-07-20 23:40:02,185 | INFO |   flush 1: 16,384/416,144 pts covered (3.9%)
2026-07-20 23:40:03,541 | INFO |   flush 2: 16,384/416,144 pts covered (3.9%)
2026-07-20 23:40:04,925 | INFO |   flush 3: 16,384/416,144 pts covered (3.9%)
2026-07-20 23:40:06,285 | INFO |   flush 4: 16,384/416,144 pts covered (3.9%)
2026-07-20 23:40:07,646 | INFO |   flush 5: 16,384/416,144 pts covered (3.9%)
2026-07-20 23:40:09,025 | INFO |   flush 6: 16,384/416,144 pts covered (3.9%)
2026-07-20 23:40:10,267 | INFO |   flush 7: 16,384/416,144 pts covered (3.9%)



───────────────────────────────────────────────────────
  File          : sample_data_0045
  Total Points  : 416,144
  Target Points : 75,487
  Other Points  : 340,657
  Target Ratio  : 18.14%
  Est. Volume   : 26.1947 m³
  95% CI        : [26.0367 – 26.5214] m³  (±0.1251)
───────────────────────────────────────────────────────


2026-07-20 23:40:33,667 | INFO | predict_full_cloud: 380,616 points, N=16,384/chunk, batch=8, device=cuda — this can take a while on CPU, progress below
2026-07-20 23:40:35,285 | INFO |   flush 1: 16,384/380,616 pts covered (4.3%)
2026-07-20 23:40:36,836 | INFO |   flush 2: 16,384/380,616 pts covered (4.3%)
2026-07-20 23:40:38,420 | INFO |   flush 3: 16,384/380,616 pts covered (4.3%)
2026-07-20 23:40:40,016 | INFO |   flush 4: 16,384/380,616 pts covered (4.3%)
2026-07-20 23:40:41,592 | INFO |   flush 5: 16,384/380,616 pts covered (4.3%)
2026-07-20 23:40:43,158 | INFO |   flush 6: 16,384/380,616 pts covered (4.3%)
2026-07-20 23:40:43,867 | INFO |   flush 7: 16,384/380,616 pts covered (4.3%)



───────────────────────────────────────────────────────
  File          : sample_data_0032
  Total Points  : 380,616
  Target Points : 45,861
  Other Points  : 334,755
  Target Ratio  : 12.05%
  Est. Volume   : 4.0096 m³
  95% CI        : [3.9564 – 4.2394] m³  (±0.0721)
───────────────────────────────────────────────────────


2026-07-20 23:43:18,960 | INFO | predict_full_cloud: 447,284 points, N=16,384/chunk, batch=8, device=cuda — this can take a while on CPU, progress below
2026-07-20 23:43:20,166 | INFO |   flush 1: 16,386/447,284 pts covered (3.7%)
2026-07-20 23:43:21,264 | INFO |   flush 2: 16,386/447,284 pts covered (3.7%)
2026-07-20 23:43:22,361 | INFO |   flush 3: 16,386/447,284 pts covered (3.7%)
2026-07-20 23:43:23,461 | INFO |   flush 4: 16,386/447,284 pts covered (3.7%)
2026-07-20 23:43:24,559 | INFO |   flush 5: 16,386/447,284 pts covered (3.7%)
2026-07-20 23:43:25,645 | INFO |   flush 6: 16,386/447,284 pts covered (3.7%)
2026-07-20 23:43:26,757 | INFO |   flush 7: 16,386/447,284 pts covered (3.7%)
2026-07-20 23:43:27,254 | INFO |   flush 8: 16,386/447,284 pts covered (3.7%)



───────────────────────────────────────────────────────
  File          : sample_data_0033
  Total Points  : 447,284
  Target Points : 88,136
  Other Points  : 359,148
  Target Ratio  : 19.70%
  Est. Volume   : 1.4363 m³
  95% CI        : [1.4114 – 1.6735] m³  (±0.0514)
───────────────────────────────────────────────────────


## Final Inference  (`data/test`, no labels)

In [ ]:
if INFER_FILES:
    log.info("=== FINAL INFERENCE ===")
    infer_names, infer_true, infer_pred, infer_pts = [], [], [], []
    for path in INFER_FILES:
        name = os.path.splitext(os.path.basename(path))[0]
        pts, preds, lbl = predict_full_cloud(model, path)
        infer_names.append(name)
        infer_true.append(lbl if lbl is not None and lbl.any() else None)
        infer_pred.append(preds)
        infer_pts.append(pts)
    per_file_report(infer_names, infer_true, infer_pred, tag="INFERENCE",
                    target_class=CONFIG["target_class"])
    for name, pts, preds in zip(infer_names, infer_pts, infer_pred):
        target_pts   = pts[preds == CONFIG["target_class"]]
        volume, conf = vol_TIN(target_pts)
        n_tot = len(pts)
        n_tgt = len(target_pts)
        print(f"\n{'─'*55}")
        print(f"  File          : {name}")
        print(f"  Total Points  : {n_tot:,}")
        print(f"  Target Points : {n_tgt:,}")
        print(f"  Other Points  : {n_tot - n_tgt:,}")
        print(f"  Target Ratio  : {n_tgt / max(n_tot, 1) * 100:.2f}%")
        print(f"  Est. Volume   : {volume:.4f} m³")
        if conf:
            print(f"  95% CI        : [{conf['ci_low']:.4f} – {conf['ci_high']:.4f}] m³"
                  f"  (±{conf['std']:.4f})")
        print(f"{'─'*55}")
        visualize_segmentation(pts, preds,
                               title=f"INFERENCE | {MODEL_NAME} | {name} | "
                                     f"V={volume:.4f} m³")
else:
    log.info("data/test is empty — skipping inference.")

2026-07-20 23:43:45,098 | INFO | === FINAL INFERENCE ===
2026-07-20 23:43:46,352 | INFO |   SOR: removed 7263 outlier pts (sample_data_0046.las)
2026-07-20 23:43:47,817 | INFO |   Cluster-filter: removed 2024 pts in small floating clusters (sample_data_0046.las)
2026-07-20 23:43:57,673 | INFO | predict_full_cloud: 461,841 points, N=16,384/chunk, batch=8, device=cuda — this can take a while on CPU, progress below
2026-07-20 23:43:58,532 | INFO |   flush 1: 16,384/461,841 pts covered (3.5%)
2026-07-20 23:43:59,297 | INFO |   flush 2: 16,384/461,841 pts covered (3.5%)
2026-07-20 23:44:00,068 | INFO |   flush 3: 16,384/461,841 pts covered (3.5%)
2026-07-20 23:44:00,834 | INFO |   flush 4: 16,384/461,841 pts covered (3.5%)
2026-07-20 23:44:01,605 | INFO |   flush 5: 16,384/461,841 pts covered (3.5%)
2026-07-20 23:44:02,371 | INFO |   flush 6: 16,384/461,841 pts covered (3.5%)
2026-07-20 23:44:03,143 | INFO |   flush 7: 16,384/461,841 pts covered (3.5%)
2026-07-20 23:44:03,659 | INFO |   flu


PER-FILE SEGMENTATION REPORT — INFERENCE
File                      IoU   Dice   Prec    Rec       TP       FP       FN       TN   GT_Wood  Pred_Wood
-----------------------------------------------------------------------------------------------------------
sample_data_0046            —      —      —      —        —        —        —        —         —    150,563
sample_data_0047            —      —      —      —        —        —        —        —         —    466,271
sample_data_0048            —      —      —      —        —        —        —        —         —    121,869
sample_data_0049            —      —      —      —        —        —        —        —         —     38,747
sample_data_0050            —      —      —      —        —        —        —        —         —    192,796
sample_data_0051            —      —      —      —        —        —        —        —         —     63,722
sample_data_0052            —      —      —      —        —        —        —        —        

In [ ]:
import torch
torch.version.cuda